**Privacy note:** Notebook outputs were cleared to avoid exposing dataset records. Configure source paths privately using environment variables.

#### Airbnb Demand Proxy Modeling in Dublin

This notebook analyzes Airbnb guest count patterns in Dublin using listing, search, and guest-contact data. Since the dataset does not contain a clean confirmed booking or nightly transaction price variable, guest count is sued as a proxy for market demand.

#### Dataset Overview

##### The analysis combines Airbnb listing, search, and contact-related data, including neighbourhood, room type, guest count, and booking-interaction variables.

#### Objective

##### The objective is to test whether listing, neighbourhood, and trip-related features can help explain variation in guest count. The project benchmarks Linear Regression and XGBoost against a navie mean baseline to evaluate whether machine learning models provide additional predictive value over a simple average prediction.

#### Scope and Limitation

##### This project should be interpreted as demand proxy modeling rather than true booking-demand forecasting or rental price prediction. The dataset does not include confirmed bookings, occupancy rates, revenue, or a clean nightly listing price field. Therefore, the model mainly to identify predictive signals and demand patterns, not to produce highly accurate business forecasts.

In [ ]:
import os
# Import libraries

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

In [ ]:
contacts = pd.read_excel(os.environ["CONTACTS_DATA_PATH"])
searches = pd.read_excel(os.environ["SEARCHES_DATA_PATH"])
result = contacts.merge(searches, left_on='id_guest', right_on='id_user', how='left')

#### Data Loading and Data Quality Checks

In [ ]:
print(result.shape)
print(result.size)
print(result.info())

#### Missing Values

The variables filter_price_min and filter_price_max contain substantial missingness, so they require careful handling before being used in analysis or modeling.

In [ ]:
missing = result.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing

#### Duplicate Records

Duplicate rows are removed only when the same guest, host, listing, check-in date, and check-out date appear more than once. This preserves legitimate repeat activity while removing exact duplicates.

In [ ]:
result['ds_checkin_x'] = pd.to_datetime(result['ds_checkin_x'], errors='coerce')
result['ds_checkout_x'] = pd.to_datetime(result['ds_checkout_x'], errors='coerce')
result = result.drop_duplicates(subset=['id_guest', 'id_host', 'id_listing', 'ds_checkin_x', 'ds_checkout_x'])

Missing values in filter_price_min are imputed with 0 because this appears to represent the default lower-bound search filter in the Airbnb interface.

In [ ]:
result['filter_price_min'] = result['filter_price_min'].fillna(0)

Missing values in filter_price_max are left as missing because many users may search without specifying an upper price limit.

In [ ]:
result['filter_price_max'].isna().sum()

Missing values in n_nights are filled with the median stay length of 3 nights. This is more reasonable than using 0 because a stay of 0 nights is not meaningful, and the distribution is right-skewed.

In [ ]:
result['n_nights_median'] = result['n_nights'].median()
result['n_nights'] = result['n_nights'].fillna(result['n_nights_median'])

#### Exploratory Data Analysis

##### Room Type Preferences

Guests search across several room types, with the strongest interest concentrated in entire homes/apartments and private rooms.

In [ ]:
result['filter_room_types'] = result['filter_room_types'].str.lstrip(',')

In [ ]:
encoded = result['filter_room_types'].str.get_dummies(sep=',')
result = result.join(encoded)

In [ ]:
result[['Entire home/apt', 'Private room', 'Shared room']].sum()

#### Neighbourhood Interest

The highest search activity is concentrated in central Dublin locations, especially areas close to the city centre.

In [ ]:
neig_encoded = result['filter_neighborhoods'].str.get_dummies(sep=',')
result = result.join(neig_encoded)

In [ ]:
result[['Ardilea', 'Ballsbridge', 'City Centre', 'Clontarf',
       'Courtlands', 'Docklands', 'Donnybrook', 'Drumcondra', 'East Wall',
       'Edenmore', 'Fairview', 'Finglas',
       'Grafton Street/St. Stephens Green',
       "North City Central/O'Connell Street", 'North Strand', 'Old City',
       'Phibsborough', 'Portobello', 'Ranelagh and Rathmines', 'Rathfarnham',
       'Smithfield', 'Temple Bar', 'Trinity College']].sum().sort_values(ascending=False)

#### Traveler Profiles

Among the longest stays, many visitors appear to come from European countries, and a large share of trips involve solo travel.

In [ ]:
result[['id_guest', 'n_guests_min', 'n_guests_max', 'n_nights', 'origin_country']].sort_values(by=['n_nights'], ascending=False).head(5)

#### Host Acceptance and Stay Length

Hosts appear more likely to accept inquiries for shorter and moderate stays, while very long stays are more often rejected.

In [ ]:
host_accepted = result[['id_host', 'id_guest', 'ts_accepted_at', 'n_searches', 'n_messages', 'n_guests', 'n_nights']]
length_stay_accepted = host_accepted.dropna(subset=['ts_accepted_at'])
length_stay_accepted.groupby(['id_host', 'n_messages', 'n_guests', 'n_searches'])['n_nights'].sum().sort_values(ascending=False)

In [ ]:
length_stay_unaccepted = host_accepted[host_accepted['ts_accepted_at'].isna()]
length_stay_unaccepted.groupby(['id_host', 'n_messages', 'n_guests', 'n_searches'])['n_nights'].sum().sort_values(ascending=False)

#### Host Acceptance and Group Size

Hosts seem more likely to accept inquiries from groups of two or more guests than from solo travelers.

In [ ]:
guests_accepted = host_accepted.dropna(subset=['ts_accepted_at'])
guests_accepted = guests_accepted.groupby(['id_host', 'n_messages', 'n_nights', 'n_searches'])['n_guests'].mean().round(0).sort_values(ascending=False)
guests_accepted.value_counts().head(5)

In [ ]:
guests_unaccepted = host_accepted[host_accepted['ts_accepted_at'].isna()]
guests_unaccepted = guests_unaccepted.groupby(['id_host', 'n_messages', 'n_nights', 'n_searches'])['n_guests'].mean().round(0).sort_values(ascending=False)
guests_unaccepted.value_counts().head(5)

#### Host Acceptance and Messaging

Hosts tend to accept inquiries that involve more message exchanges, suggesting that stronger communication may improve booking outcomes.

In [ ]:
host_messages = host_accepted.dropna(subset=['ts_accepted_at'])
host_messages = host_messages.groupby(['id_host'])['n_messages'].sum().sort_values(ascending=False)
host_messages.value_counts().head(5)

In [ ]:
hosts_messages_unaccepted = host_accepted[host_accepted['ts_accepted_at'].isna()]
hosts_messages_unaccepted = hosts_messages_unaccepted.groupby(['id_host'])['n_messages'].sum().sort_values(ascending=False)
hosts_messages_unaccepted.value_counts().head(5)

#### Distribution Analysis

#### Guest Count Distribution

Most searches involve 2 guests. The distribution is right-skewed, with frequency dropping as group size increases, which indicates that smaller groups dominate the dataset.

In [ ]:
guest_hist = result['n_guests'].hist(bins=30)
guest_hist.set_title('Distribution of Guest Count')
guest_hist.set_ylabel('Frequency')
guest_hist.set_xlabel('Number of Guests')

#### Length-of-Stay Distribution

The number of nights is strongly right-skewed. Most stays are short, while long stays occur much less frequently and appear as a small number of extreme observations.

In [ ]:
nights_hist = result['n_nights'].hist(bins=300)
nights_hist.set_title('Distribution of Length of Stay')
nights_hist.set_ylabel('Frequency')
nights_hist.set_xlabel('Number of Nights')

#### Room Type Distribution

The number of guests who preferred entire home or apartment are the highest among all the different preferences across different guests.

In [ ]:
room_type_bar = result[['Entire home/apt', 'Private room', 'Shared room']].sum()
room_type_bar = room_type_bar.plot(kind='bar', figsize=(7,4))
room_type_bar.set_xlabel('Room Type')
room_type_bar.set_ylabel('Distribution of Room Types')
room_type_bar.tick_params(axis='x', labelrotation=0)

In [ ]:
top_neigh = result[['Ardilea', 'Ballsbridge', 'City Centre', 'Clontarf', 'Courtlands', 'Docklands', 'Donnybrook', 'Drumcondra', 'East Wall',
       'Edenmore', 'Fairview', 'Finglas', 'Glasnevin', 'Grafton Street/St. Stephens Green', "North City Central/O'Connell Street", 'North Strand', 'Old City', 'Phibsborough', 'Portobello', 'Ranelagh and Rathmines', 'Rathfarnham',
       'Ringsend/Irishtown', 'Smithfield', 'Stoneybatter/Arbour Hill', 'Temple Bar', 'Trinity College']].sum()
top_neigh = top_neigh.sort_values(ascending=False).head(5)
top_neigh = top_neigh.plot(kind='bar', figsize=(12,5))
top_neigh.set_xlabel('Neighbourhood')
top_neigh.set_ylabel('Distribution of Neighbourhood Area')
top_neigh.tick_params(axis='x', labelrotation=0)

#### Predictive Modeling

This section tests whether neighbourhood, room type, stay length, and guest-related features can explain demand and price-filter behaviour.

#### Experiment 1: Predicting Minimum Price Filter

An XGBoost model is first used to estimate the minimum price filter selected by users based on neighbourhood, property type, number of guests, and stay length.

In [ ]:
X = result[['n_nights', 'n_guests', 'Entire home/apt', 'Private room', 'Shared room', 'Ardilea', 'Ballsbridge', 'City Centre', 'Clontarf', 'Courtlands', 'Docklands', 'Donnybrook', 'Drumcondra', 'East Wall',
       'Edenmore', 'Fairview', 'Finglas', 'Glasnevin', 'Grafton Street/St. Stephens Green', "North City Central/O'Connell Street", 'North Strand', 'Old City', 'Phibsborough', 'Portobello', 'Ranelagh and Rathmines', 'Rathfarnham',
       'Ringsend/Irishtown', 'Smithfield', 'Stoneybatter/Arbour Hill', 'Temple Bar', 'Trinity College']]

In [ ]:
y_min = result['filter_price_min']
y_min_mean = result['filter_price_min'].mean().round(2)
y_min = y_min.fillna(y_min_mean)
y_min = y_min.replace(0.00, y_min_mean)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y_min, test_size=0.2, random_state=42)

model_ = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)
model_.fit(X_train, y_train)
pred = model_.predict(X_val)

In [ ]:
print("Predictions:", pred[:10])
print("Actual:", y_val.iloc[:10].values)

In [ ]:
print("MAE:", round(mean_absolute_error(y_val, pred), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_val, pred)), 2))
print("R2:", round(r2_score(y_val, pred), 2))

#### Baseline Comparison for Price Filter Prediction

A simple mean baseline is used to check whether the machine learning model adds real predictive value.

In [ ]:
baseline_pred_price = np.full(len(y_val), y_train.mean())

In [ ]:
print("Baseline Prediction:", baseline_pred_price[:10])
print("Baseline Actual:", y_val.iloc[:10].values)

In [ ]:
print("Baseline MAE:", round(mean_absolute_error(y_val, baseline_pred_price), 2))
print("Baseline RMSE:", round(np.sqrt(mean_squared_error(y_val, baseline_pred_price)), 2))
print("Baseline R2 Score:", round(r2_score(y_val, baseline_pred_price), 2))

**Conclusion:** The price-filter model performs poorly. Its errors are large and the negative $R^2$ indicates that it does worse than a simple mean-based baseline.

#### Experiment 2: Predicting Guest Count

After assessing the available variables, n_guests was selected as the final regression target because the available price filter variables were noisy and unreliable. Guest count is a stronger proxy for demand in this dataset, the next model focuses on predicting the number of guests from stay and location features.

In [ ]:
X = result[['n_nights', 'Entire home/apt', 'Private room', 'Shared room', 'Ardilea', 'Ballsbridge', 'City Centre', 'Clontarf', 'Courtlands', 'Docklands', 'Donnybrook', 'Drumcondra', 'East Wall',
       'Edenmore', 'Fairview', 'Finglas', 'Glasnevin', 'Grafton Street/St. Stephens Green', "North City Central/O'Connell Street", 'North Strand', 'Old City', 'Phibsborough', 'Portobello', 'Ranelagh and Rathmines', 'Rathfarnham',
       'Ringsend/Irishtown', 'Smithfield', 'Stoneybatter/Arbour Hill', 'Temple Bar', 'Trinity College']]

In [ ]:
y = result['n_guests']

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_val)

In [ ]:
print("Predictions:", pred[:10].round(2))
print("Actual:", y_val.iloc[:10].values)

In [ ]:
print("MAE:", round(mean_absolute_error(y_val, pred), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_val, pred)), 2))
print("R2:", round(r2_score(y_val, pred), 2))

#### Baseline for Guest Count Prediction

The mean baseline provides a simple reference point for evaluating whether the model captures useful demand patterns.

In [ ]:
baseline_pred = np.full(len(y_val), y_train.mean())

In [ ]:
print("Baseline Prediction:", baseline_pred[:10].round(2))
print("Baselin Actual:", y_val.iloc[:10].values)

In [ ]:
print("Baseline MAE:", round(mean_absolute_error(y_val, baseline_pred), 2))
print("Baseline RMSE:", round(np.sqrt(mean_squared_error(y_val, baseline_pred)), 2))
print("Baseline R2:", round(r2_score(y_val, baseline_pred), 2))

#### Linear Regression Benchmark

A Linear Regression model is included as a simple and interpretable benchmark alongside XGBoost and the naive baseline.

In [ ]:
X_train_lr = X_train.fillna(0)
X_val_lr = X_val.fillna(0)
y_train_lr = y_train.fillna(0)
y_val_lr = y_val.fillna(0)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_lr, y_train_lr)
lr_pred = lr.predict(X_val_lr)

In [ ]:
print("Predictions:", lr_pred[:10].round(2))
print("Actual:", y_val.iloc[:10].values)

In [ ]:
print("Linear Regression MAE:", round(mean_absolute_error(y_val_lr, lr_pred), 2))
print("Linear Regression RMSE:", round(np.sqrt(mean_squared_error(y_val_lr, lr_pred)), 2))
print("R2 Score:", round(r2_score(y_val_lr, lr_pred), 2))

#### Model Comparison

In [ ]:
final_result = pd.DataFrame({
    "Model": ["Mean Baseline", "Linear Regression", "XGBoost"],
    "MAE": [round(mean_absolute_error(y_val, baseline_pred), 2), round(mean_absolute_error(y_val_lr, lr_pred), 2), round(mean_absolute_error(y_val, pred), 2)],
    "RMSE": [round(np.sqrt(mean_squared_error(y_val, baseline_pred)), 2), round(np.sqrt(mean_squared_error(y_val_lr, lr_pred)), 2), round(np.sqrt(mean_squared_error(y_val, pred)), 2)],
    "R2 Score": [round(r2_score(y_val, baseline_pred), 2), round(r2_score(y_val_lr, lr_pred), 2), round(r2_score(y_val, pred), 2)]
})

In [ ]:
final_result

**Conclusion:** XGBoost performs slightly better than both the mean baseline and Linear Regression when predicting guest count. However, the low $R^2$ shows that the model still explains only a small share of demand variation, so the predictive power remains limited.

#### Feature Importance for the XGBoost Model

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).head(10)

In [ ]:
importance_bar = importance.sort_values(ascending=False).head(5)
importance_bar = importance_bar.plot(kind='bar', figsize=(8,5))
importance_bar.set_ylabel('Feature Importances')
importance_bar.set_xlabel('Features')
importance_bar.tick_params(axis='x', labelrotation=0)

Room type variables such as entire home/apartment and private room are among the strongest predictors, followed by `n_nights`. Central neighbourhoods such as Temple Bar and City Centre also appear important, highlighting the combined role of accommodation type and location.

#### Error Analysis

In [ ]:
comparison = X_val.copy()
comparison['actual'] = y_val.values
comparison['pred'] = pred
comparison['abs_error'] = abs(comparison['actual'] - comparison['pred'])
comparison.sort_values("abs_error", ascending=False).head(10)

In [ ]:
comparison[['actual', 'pred', 'abs_error']].sort_values("abs_error", ascending=False).head(10)

The largest errors occur for very high actual guest counts. In those cases, the model predicts values closer to 2 to 3 guests while the true counts are much higher, suggesting that the dataset is dominated by smaller groups and the model underpredicts rare large bookings.

#### Final Takeaways

- Guest count was used as a demand proxy because the dataset did not contain confirmed bookings, occupancy, revenue, or a clean nightly listing price field
- XGBoost outperformed Linear Regression and the naive mean baseline, showing that the available features contained some predictive signal. 
- However, the low R2 score suggests that guest count variation is only weakly explained by the current features.
- This project should be interpreted as demand proxy modeling rather than true demand forecasting.
- The main value of the project is in data cleaning, feature engineering, baseline comparison, model benchmarking, and understanding the limitations of imperfect marketplace data.
- Future improvements could include confirmed bookings, occupancy rate, nightly price, seasonality, event dates, review history and host-level features.